# Aula 2 · Como o texto vira número (TF-IDF)

Um modelo de Machine Learning **não lê palavras — só entende números**.
Então como transformar `"produto ótimo"` em números que o modelo consiga usar?

A resposta é o **TF-IDF**. Neste notebook vamos construí-lo **passo a passo**,
vendo a tabela mudar a cada etapa, com um exemplo pequeno de 4 avaliações.

> Roteiro: texto cru → limpeza → contagem de palavras → IDF → TF×IDF → o que o modelo enxerga.

## Passo 0 — Nosso exemplo

Quatro avaliações curtas: duas elogiando, duas reclamando.

In [ ]:
import pandas as pd

reviews = [
    "Produto ótimo, entrega rápida",
    "Ótimo produto, recomendo",
    "Produto horrível, veio quebrado",
    "Entrega horrível, muito atraso",
]
df = pd.DataFrame({"review": reviews})
df

## Passo 1 — Limpar o texto

Antes de contar palavras, padronizamos:
- **minúsculas** → `"Ótimo"` e `"ótimo"` viram a mesma palavra
- **sem pontuação** → vírgulas e pontos não ajudam

Veja a coluna `limpo` aparecer ao lado do texto original:

In [ ]:
df["limpo"] = (
    df["review"]
      .str.lower()
      .str.replace(r"[^a-zà-ú ]", "", regex=True)
)
df[["review", "limpo"]]

## Passo 2 — Bag of Words: contar palavras

A ideia mais simples: para cada avaliação, **contar quantas vezes cada palavra aparece**.

Isso vira uma tabela onde:
- cada **linha** é uma avaliação
- cada **coluna** é uma palavra do vocabulário
- cada **número** é a contagem

Chamamos isso de *Bag of Words* (saco de palavras) — perde a ordem, guarda a presença.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
contagem = cv.fit_transform(df["limpo"])

bow = pd.DataFrame(contagem.toarray(), columns=cv.get_feature_names_out())
bow.index = [f"review {i}" for i in range(len(df))]
bow

Olhe a palavra **`produto`**: aparece em 3 das 4 avaliações.
E **`recomendo`**: aparece em 1 só.

Aí mora o problema do Bag of Words puro: ele dá o mesmo peso (1) para `produto`
(comum, pouco informativa) e `recomendo` (rara, muito informativa).
Palavras comuns acabam dominando.

> E se uma palavra aparecesse em **todas** as avaliações? Ela não ajudaria a
> diferenciar nenhuma — o TF-IDF vai dar a ela um peso quase zero, descartando
> palavras inúteis **sozinho**, sem precisar de uma lista de "palavras a ignorar".

## Passo 3 — TF-IDF: dar mais peso às palavras raras

O TF-IDF resolve isso. O nome são duas partes:

- **TF (Term Frequency)** — **quantas vezes** a palavra aparece *naquela* avaliação.
  Falar "ótimo" duas vezes pesa mais que falar uma.
- **IDF (Inverse Document Frequency)** — quão **rara** a palavra é no conjunto todo.

**Analogia para o IDF:** imagine uma festa. Se quase todo mundo usa camiseta preta
(`produto`, que aparece em quase todas as avaliações), a camiseta preta não ajuda a
distinguir ninguém. Já o chapéu vermelho (`recomendo`, que aparece numa só) chama
atenção e distingue. O IDF dá **nota alta para os chapéus vermelhos** e **nota baixa
para as camisetas pretas**.

O peso final é **TF × IDF**: alto quando a palavra é frequente no texto **e** rara no geral.

> Detalhe do nosso exemplo: nenhuma palavra se repete dentro de uma frase, então o
> TF é sempre 1. Aqui quem manda é o IDF. Em textos maiores, o TF entra em jogo.

### Primeiro, veja o IDF de cada palavra

Quanto mais rara a palavra, maior o IDF (maior o "prêmio").

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer()
tfidf = tv.fit_transform(df["limpo"])

idf = (pd.DataFrame({"palavra": tv.get_feature_names_out(), "idf": tv.idf_.round(2)})
         .sort_values("idf", ascending=False))
idf

Repare: `produto` (em 3 de 4 avaliações) tem o **menor** IDF — é a "camiseta preta".
As palavras raras (`atraso`, `recomendo`, `quebrado`...) têm o **maior** — são os "chapéus vermelhos".

### Vendo a multiplicação TF × IDF

Vamos pegar a `review 1` ("ótimo produto recomendo") e fazer a conta na mão para
duas palavras — uma comum e uma rara:

In [ ]:
idf_produto = float(idf.loc[idf["palavra"] == "produto", "idf"].iloc[0])
idf_recomendo = float(idf.loc[idf["palavra"] == "recomendo", "idf"].iloc[0])

# Na review 1, cada palavra aparece 1 vez -> TF = 1
print(f"produto:   TF=1  x  IDF={idf_produto}  =  {1*idf_produto:.2f}   (comum  -> peso baixo)")
print(f"recomendo: TF=1  x  IDF={idf_recomendo}  =  {1*idf_recomendo:.2f}   (rara   -> peso alto)")

## Passo 4 — A matriz TF-IDF final

Mesma tabela do Bag of Words, mas com **pesos** no lugar de contagens.
Compare com o Passo 2: as palavras raras e informativas ganham os maiores pesos
**dentro de cada avaliação**, enquanto `produto` (comum) encolhe.

> Os números não são as contagens redondas que calculamos acima: o TF-IDF
> **reescala cada linha** para que as avaliações fiquem comparáveis entre si.
> Não se prenda ao valor exato — o que importa é a **ordem de importância dentro da linha**.

In [ ]:
mat = pd.DataFrame(tfidf.toarray().round(2), columns=tv.get_feature_names_out())
mat.index = [f"review {i}" for i in range(len(df))]
mat

### A palavra mais forte de cada avaliação

Este é o resultado que importa: em cada avaliação, qual palavra o modelo considera
mais distintiva? Repare que `produto` (comum) **nunca** ganha.

In [ ]:
for i in range(len(df)):
    linha = mat.iloc[i]
    palavra = linha.idxmax()
    print(f"review {i}: palavra mais forte = '{palavra}' ({linha.max():.2f})  <-  {df['limpo'][i]}")

## Passo 5 — É isso que o modelo recebe

Cada avaliação virou uma **linha de números** (uma coluna para cada palavra do vocabulário).
O modelo aprende quais palavras puxam para "satisfeito" e quais puxam para "insatisfeito".

No dataset real do Olist, em vez de 10 palavras, o vocabulário tem **milhares** —
a matriz fica enorme, mas a ideia é exatamente esta.

In [ ]:
import urllib.request, pathlib
URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
P = pathlib.Path("olist.parquet")
if not P.exists():
    urllib.request.urlretrieve(URL, P)

reais = pd.read_parquet(P, columns=["review_comment_message"])["review_comment_message"].dropna()
reais = reais[reais.str.len() > 0].str.lower().str.replace(r"[^a-zà-ú ]", "", regex=True)

tv_real = TfidfVectorizer(min_df=3)
m = tv_real.fit_transform(reais)
print(f"Avaliações:                       {m.shape[0]:,}")
print(f"Palavras no vocabulário (colunas): {m.shape[1]:,}")

> Boa notícia: você **não precisa** fazer isso à mão. Dentro de um `Pipeline` do
> scikit-learn, o `TfidfVectorizer` faz todos esses passos automaticamente. Mas agora
> você sabe o que acontece por baixo do capô.

## Recapitulando

| Etapa | O que fizemos |
|-------|---------------|
| 0 | Texto cru |
| 1 | Limpeza (minúsculas, sem pontuação) |
| 2 | Bag of Words — contar palavras |
| 3 | IDF — medir o quão rara é cada palavra |
| 4 | TF × IDF — pesar palavras raras e informativas |
| 5 | Cada avaliação vira uma linha de números que o modelo entende |

**A frase que resume tudo:** TF-IDF é *"contar palavras, dando mais peso às raras e informativas"*.

No próximo notebook (`02-do-texto-ao-modelo`) usamos isso dentro de um modelo que junta
o **texto** com os dados de **tabela** (preço, prazo, frete) — para classificar
TODOS os pedidos, tenham eles comentário ou não.